In [ ]:
pip install rdkit

In [ ]:
! pip install padelpy

In [ ]:
! wget https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip
! unzip fingerprints_xml.zip

In [ ]:
import glob
xml_files = glob.glob("*.xml")
xml_files.sort()
xml_files

In [ ]:
FP_list = ['AtomPairs2DCount',
 'AtomPairs2D',
 'EState',
 'CDKextended',
 'CDK',
 'CDKgraphonly',
 'KlekotaRothCount',
 'KlekotaRoth',
 'MACCS',
 'PubChem',
 'SubstructureCount',
 'Substructure']

In [ ]:
fp = dict(zip(FP_list, xml_files))
fp

In [ ]:
fp['AtomPairs2D']

In [ ]:
import pandas as pd
df = pd.read_csv('Kinaset.csv')
df.head(2)

In [ ]:
df2 = pd.concat( [df['Smiles'],df['Molecule']], axis=1 )
df2.to_csv('molecule.smi', sep='\t', index=False, header=False)
df2

In [ ]:
fp

In [ ]:
from padelpy import padeldescriptor

fingerprint = 'Substructure'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml',
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

In [ ]:
from padelpy import padeldescriptor
fingerprints = {
    'AtomPairs2D': 'AtomPairs2DFingerprinter.xml',
    'AtomPairs2DCount': 'AtomPairs2DFingerprintCount.xml',
    'CDK': 'Fingerprinter.xml',
    'CDKextended': 'ExtendedFingerprinter.xml',
    'CDKgraphonly': 'GraphOnlyFingerprinter.xml',
    'EState': 'EStateFingerprinter.xml',
    'KlekotaRoth': 'KlekotaRothFingerprinter.xml',
    'KlekotaRothCount': 'KlekotaRothFingerprintCount.xml',
    'MACCS': 'MACCSFingerprinter.xml',
    'PubChem': 'PubchemFingerprinter.xml',
    'Substructure': 'SubstructureFingerprinter.xml',
    'SubstructureCount': 'SubstructureFingerprintCount.xml'
}
mol_dir = 'molecule.smi'
for fingerprint, descriptor_type in fingerprints.items():
    fingerprint_output_file = f'{fingerprint}.csv'
    fingerprint_descriptortypes = descriptor_type

    padeldescriptor(mol_dir=mol_dir,
                    d_file=fingerprint_output_file,
                    descriptortypes=fingerprint_descriptortypes,
                    detectaromaticity=True,
                    standardizenitro=True,
                    standardizetautomers=True,
                    threads=2,
                    removesalt=True,
                    log=True,
                    fingerprints=True)

In [ ]:
import pandas as pd

df3 = pd.read_csv('AtomPairs2DCount.csv')
df3

In [ ]:
df4 = pd.read_csv('AtomPairs2D.csv')
df4

In [ ]:
df5 = pd.read_csv('CDK.csv')
df5

In [ ]:
df6 = pd.read_csv('CDKextended.csv')
df6

In [ ]:
df7 = pd.read_csv('EState.csv')
df7

In [ ]:
df8 = pd.read_csv('KlekotaRoth.csv')
df8

In [ ]:
df9 = pd.read_csv('KlekotaRothCount.csv')
df9

In [ ]:
df10 = pd.read_csv('Substructure.csv')
df10

In [ ]:
df11 = pd.read_csv('SubstructureCount.csv')
df11

In [ ]:
df12 = pd.read_csv('Pubchem.csv')
df12

In [ ]:
df13 = pd.read_csv('MACCS.csv')
df13

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

def calculate_morgan_fingerprint(smiles, radius=2, nBits=2048):

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    return list(fp)

df_smi = pd.read_csv('molecule.smi', sep='\t', header=None, names=['canonical_smiles', 'molecule_chembl_id'])

df_smi['Morgan_FP'] = df_smi['canonical_smiles'].apply(lambda x: calculate_morgan_fingerprint(x))

fingerprint_columns = [f'Bit_{i}' for i in range(2048)]  # Adjust the range if nBits is changed
df_fp = df_smi['Morgan_FP'].apply(pd.Series)
df_fp.columns = fingerprint_columns

# Combine with original data
df_morgan = pd.concat([df_smi.drop(columns=['Morgan_FP']), df_fp], axis=1)

# Save to a CSV file
df_morgan.to_csv('morgan_fingerprints.csv', index=False)
print("Morgan fingerprints calculated and saved to 'morgan_fingerprints.csv'")


In [ ]:
final_dataset = pd.concat([df13,df12,df11,df10,df9,df8,df7,df6,df5,df4,df3,df_morgan], axis=1)

In [ ]:
final_dataset

In [ ]:
final_dataset.to_csv('final_dataset.csv', index=False)

In [ ]:
# repeat for library_files = ["KINASET.csv", "KINACORE.csv", "KINASE3D.csv", "ENAMINE.csv", "DRUGBANK.csv"]